# Classical dynamics and optimization

This notebook shows two differentiable workflows in `neurosim`:

- a chaotic double-pendulum simulation
- a launch-speed optimization problem using the projectile helper


The goal is to stay inside the actual public API. Everything below
uses `neurosim` objects that are available from the package root.

In [ ]:
import jax
import jax.numpy as jnp
import neurosim as ns

jax.config.update("jax_enable_x64", True)


In [ ]:
def lagrangian(q: jnp.ndarray, qdot: jnp.ndarray, params: ns.Params) -> jnp.ndarray:
    theta1, theta2 = q
    omega1, omega2 = qdot
    m1, m2, l1, l2, g = params.m1, params.m2, params.l1, params.l2, params.g
    kinetic = 0.5 * m1 * (l1 * omega1) ** 2 + 0.5 * m2 * (
        (l1 * omega1) ** 2 + (l2 * omega2) ** 2 + 2 * l1 * l2 * omega1 * omega2 * jnp.cos(theta1 - theta2)
    )
    potential = -(m1 + m2) * g * l1 * jnp.cos(theta1) - m2 * g * l2 * jnp.cos(theta2)
    return kinetic - potential

system = ns.LagrangianSystem(lagrangian, n_dof=2)
params = ns.Params(m1=1.0, m2=1.0, l1=1.0, l2=1.0, g=9.81)
trajectory = system.simulate(
    q0=[jnp.pi / 4, jnp.pi / 2],
    qdot0=[0.0, 0.0],
    t_span=(0.0, 10.0),
    dt=0.002,
    params=params,
    integrator="rk4",
    save_every=10,
)
trajectory.energy_drift()


In [ ]:
target_range = 1200.0

def miss_distance(v0: float | jnp.ndarray) -> jnp.ndarray:
    return (ns.projectile(v0=v0, angle=35.0, g=1.62).range - target_range) ** 2

result = ns.optimize(miss_distance, initial_guess=200.0, learning_rate=0.05, method="adam")
result.x, result.fun, result.converged


The two examples capture the general pattern used throughout the
repository: write the physics model as a pure function, then feed it
into the solver or optimizer.